In [ ]:
"""
Commodity / Sector Correlation Analysis
======================================

Purpose
-------
This script downloads historical price data for a custom universe of:
- Metals
- Energy
- Soft agricultural commodities
- Shipping equities
- Insurance equities

It then performs:
1. Asset-level daily return correlation
2. Sector-level daily return correlation
3. Full-period correlation of each asset with oil
4. Rolling correlation of each asset with oil
5. Full-period correlation of each sector with oil
6. Rolling correlation of each sector with oil

Outputs
-------
The script saves:
- An Excel workbook with raw prices, returns, and correlation tables
- Heatmap PNGs for asset and sector correlations
- Rolling-correlation PNGs vs oil

Typical Use Case
----------------
This is useful for the kind of analysis discussed in your meeting:
- Check whether metals / agri / shipping / insurance move with oil
- See whether correlations are stable or regime-dependent
- Identify sectors/assets that may be useful as related trades, hedges, or proxies

Requirements
------------
Install these libraries if needed:
    pip install pandas numpy yfinance matplotlib openpyxl

How to Use
----------
1. Update START_DATE and OUTPUT_FOLDER
2. Update the ticker universe if you want
3. Run the script
4. Check the Excel file and charts in the output folder

Important Notes
---------------
- Commodity tickers below are Yahoo Finance futures tickers
- Shipping and insurance are represented using listed equity proxies
- Sector returns are equal-weighted averages of member returns
- Correlation is based on daily percentage returns, not prices
"""

import os
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# CONFIG
# ============================================================

# Start date for historical download
START_DATE = "2018-01-01"

# End date for historical download
# Use None to download till latest available date
END_DATE = None

# Output folder
OUTPUT_FOLDER = r"D:\work\Trade Analysis\Commodity_Correlation_Analysis"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Oil benchmark label used throughout correlation analysis
OIL_LABEL = "WTI_Oil"

# Rolling windows for dynamic correlation analysis
ROLLING_WINDOWS = [20, 60, 120]

# Minimum number of non-null observations required in a column
# If a series has too few valid points, it is dropped
MIN_VALID_OBS = 30

# ------------------------------------------------------------
# TICKER UNIVERSE
# ------------------------------------------------------------
# You can modify these names/tickers as needed.
#
# Commodity futures examples on Yahoo:
# Gold     = GC=F
# Silver   = SI=F
# Copper   = HG=F
# WTI Oil  = CL=F
# Brent    = BZ=F
# NatGas   = NG=F
# Corn     = ZC=F
# Wheat    = ZW=F
# Soybeans = ZS=F
# Coffee   = KC=F
# Sugar    = SB=F
# Cocoa    = CC=F
# Cotton   = CT=F
# ------------------------------------------------------------

SECTOR_TICKERS = {
    "Metals": {
        "Gold": "GC=F",
        "Silver": "SI=F",
        "Copper": "HG=F",
    },
    "Energy": {
        "WTI_Oil": "CL=F",
        "Brent_Oil": "BZ=F",
        "NatGas": "NG=F",
    },
    "Soft_Agri": {
        "Corn": "ZC=F",
        "Wheat": "ZW=F",
        "Soybeans": "ZS=F",
        "Coffee": "KC=F",
        "Sugar": "SB=F",
        "Cocoa": "CC=F",
        "Cotton": "CT=F",
    },
    "Shipping_Equities": {
        "ZIM": "ZIM",
        "MATX": "MATX",
        "SBLK": "SBLK",
        "GOGL": "GOGL",
        "FRO": "FRO",
        "STNG": "STNG",
    },
    "Insurance_Equities": {
        "TRV": "TRV",
        "CB": "CB",
        "ALL": "ALL",
        "AIG": "AIG",
        "MMC": "MMC",
    }
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def flatten_sector_dict(sector_dict):
    """
    Convert nested dictionary structure into:
    1. flat_map : {label: ticker}
    2. meta_df  : DataFrame with sector / label / ticker mapping

    Parameters
    ----------
    sector_dict : dict
        Nested dict in format:
        {
            "Sector1": {"LabelA": "TickerA", "LabelB": "TickerB"},
            "Sector2": {"LabelC": "TickerC"}
        }

    Returns
    -------
    flat_map : dict
    meta_df : pd.DataFrame
    """
    rows = []
    flat_map = {}

    for sector, assets in sector_dict.items():
        for label, ticker in assets.items():
            flat_map[label] = ticker
            rows.append({
                "Sector": sector,
                "Label": label,
                "Ticker": ticker
            })

    meta_df = pd.DataFrame(rows)
    return flat_map, meta_df


def download_prices(label_to_ticker, start_date, end_date=None):
    """
    Download daily prices from Yahoo Finance.

    Logic:
    - Try 'Adj Close' if available
    - Otherwise use 'Close'
    - Return a wide DataFrame with one column per asset label

    Parameters
    ----------
    label_to_ticker : dict
        Mapping of user label to Yahoo ticker
    start_date : str
        e.g. '2018-01-01'
    end_date : str or None
        e.g. '2025-12-31' or None

    Returns
    -------
    prices : pd.DataFrame
        Index = date
        Columns = asset labels
    """
    ticker_list = list(label_to_ticker.values())

    print(f"Downloading {len(ticker_list)} tickers from Yahoo Finance...")

    raw = yf.download(
        tickers=ticker_list,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False,
        group_by="ticker",
        threads=True
    )

    if raw.empty:
        raise ValueError("No data was downloaded. Please check internet/tickers/date range.")

    price_dict = {}

    for label, ticker in label_to_ticker.items():
        try:
            if isinstance(raw.columns, pd.MultiIndex):
                if ticker not in raw.columns.get_level_values(0):
                    print(f"[WARNING] Ticker missing from download output: {label} ({ticker})")
                    continue

                ticker_df = raw[ticker].copy()

                if "Adj Close" in ticker_df.columns:
                    ser = ticker_df["Adj Close"].copy()
                elif "Close" in ticker_df.columns:
                    ser = ticker_df["Close"].copy()
                else:
                    print(f"[WARNING] No Adj Close / Close found for {label} ({ticker})")
                    continue

            else:
                # Single ticker fallback case
                if "Adj Close" in raw.columns:
                    ser = raw["Adj Close"].copy()
                elif "Close" in raw.columns:
                    ser = raw["Close"].copy()
                else:
                    print(f"[WARNING] No Adj Close / Close found for {label} ({ticker})")
                    continue

            ser.name = label
            price_dict[label] = ser

        except Exception as e:
            print(f"[ERROR] Failed for {label} ({ticker}): {e}")

    if not price_dict:
        raise ValueError("No usable price series downloaded successfully.")

    prices = pd.concat(price_dict.values(), axis=1).sort_index()
    return prices


def clean_price_data(prices, min_valid_obs=30):
    """
    Basic cleanup of downloaded price data.

    Steps
    -----
    1. Sort by date
    2. Remove columns with too few valid observations
    3. Remove rows where all values are NaN

    Parameters
    ----------
    prices : pd.DataFrame
    min_valid_obs : int

    Returns
    -------
    prices : pd.DataFrame
    """
    prices = prices.sort_index()

    valid_counts = prices.notna().sum()
    keep_cols = valid_counts[valid_counts >= min_valid_obs].index.tolist()

    dropped = sorted(set(prices.columns) - set(keep_cols))
    if dropped:
        print(f"[INFO] Dropping columns with < {min_valid_obs} valid observations: {dropped}")

    prices = prices[keep_cols].copy()
    prices = prices.dropna(how="all")

    if prices.empty:
        raise ValueError("No valid price data remains after cleanup.")

    return prices


def calculate_returns(prices):
    """
    Compute daily percentage returns.

    Parameters
    ----------
    prices : pd.DataFrame

    Returns
    -------
    returns : pd.DataFrame
    """
    returns = prices.pct_change()
    returns = returns.replace([np.inf, -np.inf], np.nan)
    returns = returns.dropna(how="all")
    return returns


def sector_level_returns(returns, meta_df):
    """
    Build equal-weight sector return series.

    For each sector:
    - find all its asset labels
    - keep labels present in returns
    - compute equal-weight average return across those labels

    Parameters
    ----------
    returns : pd.DataFrame
    meta_df : pd.DataFrame

    Returns
    -------
    sector_ret : pd.DataFrame
        Index = dates
        Columns = sectors
    """
    sector_ret = pd.DataFrame(index=returns.index)

    for sector in meta_df["Sector"].unique():
        labels = meta_df.loc[meta_df["Sector"] == sector, "Label"].tolist()
        valid_labels = [c for c in labels if c in returns.columns]

        if len(valid_labels) == 0:
            print(f"[WARNING] No valid assets found for sector: {sector}")
            continue

        sector_ret[sector] = returns[valid_labels].mean(axis=1, skipna=True)

    sector_ret = sector_ret.dropna(how="all")
    return sector_ret


def full_period_corr_with_oil(returns, oil_label):
    """
    Compute full-period correlation of every asset with oil.

    Parameters
    ----------
    returns : pd.DataFrame
    oil_label : str

    Returns
    -------
    oil_corr_table : pd.DataFrame
    """
    if oil_label not in returns.columns:
        raise ValueError(
            f"{oil_label} not found in returns columns.\n"
            f"Available columns: {list(returns.columns)}"
        )

    corr_series = returns.corr()[oil_label]
    oil_corr_table = pd.DataFrame({
        "Full_Period_Corr_With_Oil": corr_series
    }).sort_values("Full_Period_Corr_With_Oil", ascending=False)

    return oil_corr_table


def rolling_corrs_with_oil(returns, oil_label, windows):
    """
    Compute rolling correlation of each asset with oil.

    Parameters
    ----------
    returns : pd.DataFrame
    oil_label : str
    windows : list[int]

    Returns
    -------
    rolling_corrs : dict
        {window: DataFrame(index=dates, columns=assets)}
    latest_table : pd.DataFrame
        Summary table with latest rolling values
    """
    if oil_label not in returns.columns:
        raise ValueError(
            f"{oil_label} not found in returns columns.\n"
            f"Available columns: {list(returns.columns)}"
        )

    latest_table = pd.DataFrame(index=returns.columns)
    latest_table["Full_Period_Corr_With_Oil"] = returns.corr()[oil_label]

    rolling_corrs = {}

    oil_series = returns[oil_label]

    for w in windows:
        rc = pd.DataFrame(index=returns.index)

        for col in returns.columns:
            rc[col] = returns[col].rolling(w).corr(oil_series)

        rolling_corrs[w] = rc

        latest_vals = {}
        for col in rc.columns:
            ser = rc[col].dropna()
            latest_vals[col] = ser.iloc[-1] if not ser.empty else np.nan

        latest_table[f"Last_{w}D_Rolling_Corr_With_Oil"] = pd.Series(latest_vals)

    latest_table = latest_table.sort_values("Full_Period_Corr_With_Oil", ascending=False)

    return rolling_corrs, latest_table


def sector_corrs_with_oil(sector_returns, asset_returns, oil_label, windows):
    """
    Compute sector-level full-period and rolling correlation with oil.

    Parameters
    ----------
    sector_returns : pd.DataFrame
    asset_returns : pd.DataFrame
    oil_label : str
    windows : list[int]

    Returns
    -------
    sector_vs_oil : pd.DataFrame
        One row per sector
    """
    if oil_label not in asset_returns.columns:
        raise ValueError(
            f"{oil_label} not found in asset returns columns.\n"
            f"Available columns: {list(asset_returns.columns)}"
        )

    oil_series = asset_returns[oil_label]
    sector_vs_oil = pd.DataFrame(index=sector_returns.columns)

    # Full-period correlation
    for sector in sector_returns.columns:
        sector_vs_oil.loc[sector, "Full_Period_Corr_With_Oil"] = (
            sector_returns[sector].corr(oil_series)
        )

    # Latest rolling correlation for each sector
    for w in windows:
        latest_vals = {}

        for sector in sector_returns.columns:
            roll_corr = sector_returns[sector].rolling(w).corr(oil_series)
            roll_corr = roll_corr.dropna()
            latest_vals[sector] = roll_corr.iloc[-1] if not roll_corr.empty else np.nan

        sector_vs_oil[f"Last_{w}D_Rolling_Corr_With_Oil"] = pd.Series(latest_vals)

    sector_vs_oil = sector_vs_oil.sort_values("Full_Period_Corr_With_Oil", ascending=False)
    return sector_vs_oil


def save_corr_heatmap(corr_df, title, out_path):
    """
    Save a basic correlation heatmap using matplotlib.

    Parameters
    ----------
    corr_df : pd.DataFrame
    title : str
    out_path : str
    """
    if corr_df.empty:
        print(f"[WARNING] Skipping heatmap because DataFrame is empty: {title}")
        return

    plt.figure(figsize=(12, 10))
    plt.imshow(corr_df, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=90)
    plt.yticks(range(len(corr_df.index)), corr_df.index)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def plot_rolling_corrs_vs_oil(rolling_corr_df, oil_label, out_path, top_n=10):
    """
    Plot rolling correlation vs oil for the top assets based on latest
    absolute rolling correlation value.

    Parameters
    ----------
    rolling_corr_df : pd.DataFrame
    oil_label : str
    out_path : str
    top_n : int
    """
    if rolling_corr_df.empty:
        print(f"[WARNING] Rolling correlation DataFrame is empty: {out_path}")
        return

    latest_vals = rolling_corr_df.iloc[-1].dropna()

    if latest_vals.empty:
        print(f"[WARNING] No latest rolling correlation values available: {out_path}")
        return

    candidates = latest_vals.drop(labels=[oil_label], errors="ignore")
    if candidates.empty:
        print(f"[WARNING] No candidate assets to plot vs oil: {out_path}")
        return

    top_assets = candidates.abs().sort_values(ascending=False).head(top_n).index.tolist()

    plt.figure(figsize=(14, 7))
    for col in top_assets:
        plt.plot(rolling_corr_df.index, rolling_corr_df[col], label=col)

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(f"Rolling Correlation vs {oil_label}")
    plt.legend(loc="best", ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def build_data_availability_table(prices, meta_df):
    """
    Build a small QC table showing:
    - sector
    - label
    - ticker
    - number of valid observations
    - first available date
    - last available date

    Parameters
    ----------
    prices : pd.DataFrame
    meta_df : pd.DataFrame

    Returns
    -------
    qc_df : pd.DataFrame
    """
    rows = []

    for _, r in meta_df.iterrows():
        label = r["Label"]
        if label not in prices.columns:
            rows.append({
                "Sector": r["Sector"],
                "Label": label,
                "Ticker": r["Ticker"],
                "Valid_Obs": 0,
                "First_Date": pd.NaT,
                "Last_Date": pd.NaT,
                "Present_In_Prices": "No"
            })
        else:
            ser = prices[label].dropna()
            rows.append({
                "Sector": r["Sector"],
                "Label": label,
                "Ticker": r["Ticker"],
                "Valid_Obs": len(ser),
                "First_Date": ser.index.min() if not ser.empty else pd.NaT,
                "Last_Date": ser.index.max() if not ser.empty else pd.NaT,
                "Present_In_Prices": "Yes"
            })

    qc_df = pd.DataFrame(rows)
    return qc_df.sort_values(["Sector", "Label"]).reset_index(drop=True)


# ============================================================
# MAIN EXECUTION
# ============================================================

def main():
    print("=" * 80)
    print("STEP 1: Build universe mapping")
    print("=" * 80)
    flat_map, meta_df = flatten_sector_dict(SECTOR_TICKERS)
    print(meta_df)

    print("\n" + "=" * 80)
    print("STEP 2: Download prices")
    print("=" * 80)
    prices = download_prices(flat_map, START_DATE, END_DATE)

    print("\nDownloaded price columns:")
    print(list(prices.columns))

    print("\n" + "=" * 80)
    print("STEP 3: Clean price data")
    print("=" * 80)
    prices = clean_price_data(prices, min_valid_obs=MIN_VALID_OBS)

    print(f"Final price shape: {prices.shape}")
    print(f"Date range: {prices.index.min()} to {prices.index.max()}")

    qc_df = build_data_availability_table(prices, meta_df)

    print("\n" + "=" * 80)
    print("STEP 4: Calculate daily returns")
    print("=" * 80)
    returns = calculate_returns(prices)

    print(f"Returns shape: {returns.shape}")

    if OIL_LABEL not in returns.columns:
        raise ValueError(
            f"Configured oil label '{OIL_LABEL}' is not present in returns.\n"
            f"Available columns: {list(returns.columns)}"
        )

    print("\n" + "=" * 80)
    print("STEP 5: Asset-level correlation matrix")
    print("=" * 80)
    asset_corr = returns.corr()

    print("\n" + "=" * 80)
    print("STEP 6: Sector-level return construction")
    print("=" * 80)
    sector_returns = sector_level_returns(returns, meta_df)
    sector_returns = sector_returns.dropna(how="all")

    print(f"Sector returns shape: {sector_returns.shape}")
    print(f"Sectors available: {list(sector_returns.columns)}")

    print("\n" + "=" * 80)
    print("STEP 7: Sector-level correlation matrix")
    print("=" * 80)
    sector_corr = sector_returns.corr()

    print("\n" + "=" * 80)
    print("STEP 8: Asset correlation with oil")
    print("=" * 80)
    rolling_corrs, oil_corr_table = rolling_corrs_with_oil(
        returns=returns,
        oil_label=OIL_LABEL,
        windows=ROLLING_WINDOWS
    )

    print(oil_corr_table.head(10))

    print("\n" + "=" * 80)
    print("STEP 9: Sector correlation with oil")
    print("=" * 80)
    sector_vs_oil = sector_corrs_with_oil(
        sector_returns=sector_returns,
        asset_returns=returns,
        oil_label=OIL_LABEL,
        windows=ROLLING_WINDOWS
    )

    print(sector_vs_oil)

    print("\n" + "=" * 80)
    print("STEP 10: Save Excel workbook")
    print("=" * 80)
    excel_path = os.path.join(OUTPUT_FOLDER, "commodity_sector_correlation_analysis.xlsx")

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        meta_df.to_excel(writer, sheet_name="Universe", index=False)
        qc_df.to_excel(writer, sheet_name="Data_Availability_QC", index=False)
        prices.to_excel(writer, sheet_name="Prices")
        returns.to_excel(writer, sheet_name="Returns")
        asset_corr.to_excel(writer, sheet_name="Asset_Correlation")
        sector_returns.to_excel(writer, sheet_name="Sector_Returns")
        sector_corr.to_excel(writer, sheet_name="Sector_Correlation")
        oil_corr_table.to_excel(writer, sheet_name="Oil_Correlation_Table")
        sector_vs_oil.to_excel(writer, sheet_name="Sector_vs_Oil")

        for w, rc in rolling_corrs.items():
            sheet_name = f"RollCorr_Oil_{w}D"
            rc.to_excel(writer, sheet_name=sheet_name)

    print(f"Excel saved successfully:\n{excel_path}")

    print("\n" + "=" * 80)
    print("STEP 11: Save charts")
    print("=" * 80)

    save_corr_heatmap(
        corr_df=asset_corr,
        title="Asset Return Correlation Heatmap",
        out_path=os.path.join(OUTPUT_FOLDER, "asset_correlation_heatmap.png")
    )

    save_corr_heatmap(
        corr_df=sector_corr,
        title="Sector Return Correlation Heatmap",
        out_path=os.path.join(OUTPUT_FOLDER, "sector_correlation_heatmap.png")
    )

    for w, rc in rolling_corrs.items():
        plot_rolling_corrs_vs_oil(
            rolling_corr_df=rc,
            oil_label=OIL_LABEL,
            out_path=os.path.join(OUTPUT_FOLDER, f"rolling_corr_vs_oil_{w}d.png"),
            top_n=10
        )

    print("Charts saved successfully.")

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)
    print("Generated outputs:")
    print(f"- Excel workbook : {excel_path}")
    print(f"- Charts folder  : {OUTPUT_FOLDER}")


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

STEP 1: Build universe mapping
                Sector      Label Ticker
0               Metals       Gold   GC=F
1               Metals     Silver   SI=F
2               Metals     Copper   HG=F
3               Energy    WTI_Oil   CL=F
4               Energy  Brent_Oil   BZ=F
5               Energy     NatGas   NG=F
6            Soft_Agri       Corn   ZC=F
7            Soft_Agri      Wheat   ZW=F
8            Soft_Agri   Soybeans   ZS=F
9            Soft_Agri     Coffee   KC=F
10           Soft_Agri      Sugar   SB=F
11           Soft_Agri      Cocoa   CC=F
12           Soft_Agri     Cotton   CT=F
13   Shipping_Equities        ZIM    ZIM
14   Shipping_Equities       MATX   MATX
15   Shipping_Equities       SBLK   SBLK
16   Shipping_Equities       GOGL   GOGL
17   Shipping_Equities        FRO    FRO
18   Shipping_Equities       STNG   STNG
19  Insurance_Equities        TRV    TRV
20  Insurance_Equities         CB     CB
21  Insurance_Equities        ALL    ALL
22  Insurance_Equities    

HTTP Error 404: 

1 Failed download:
['GOGL']: YFTzMissingError('possibly delisted; no timezone found')



Downloaded price columns:
['Gold', 'Silver', 'Copper', 'WTI_Oil', 'Brent_Oil', 'NatGas', 'Corn', 'Wheat', 'Soybeans', 'Coffee', 'Sugar', 'Cocoa', 'Cotton', 'ZIM', 'MATX', 'SBLK', 'GOGL', 'FRO', 'STNG', 'TRV', 'CB', 'ALL', 'AIG', 'MMC']

STEP 3: Clean price data
[INFO] Dropping columns with < 30 valid observations: ['GOGL']
Final price shape: (2061, 23)
Date range: 2018-01-02 00:00:00 to 2026-03-11 00:00:00

STEP 4: Calculate daily returns
Returns shape: (2060, 23)

STEP 5: Asset-level correlation matrix

STEP 6: Sector-level return construction
Sector returns shape: (2060, 5)
Sectors available: ['Metals', 'Energy', 'Soft_Agri', 'Shipping_Equities', 'Insurance_Equities']

STEP 7: Sector-level correlation matrix

STEP 8: Asset correlation with oil
           Full_Period_Corr_With_Oil  Last_20D_Rolling_Corr_With_Oil  \
WTI_Oil                     1.000000                        1.000000   
Brent_Oil                   0.472355                        0.938984   
SBLK                       

: 